# 📓 Documentación Técnica — API Extracción de Datos (Mikels)

Notebook de referencia personal. Cubre, de punta a punta:

1. Arquitectura general del proyecto
2. La consulta SQL base (`sql_scripts/extraccion.sql`) y su lógica de negocio
3. Cómo `main/queries.py` arma las consultas dinámicamente
4. El pool de conexiones (`main/database.py`)
5. Los endpoints de la API (`main/main.py`)
6. Autenticación JWT (desactivada, lista para reactivar)
7. Pipeline de despliegue: Docker → GitHub Actions → GHCR → Azure Container Apps
8. El problema de red (NSG) y cómo se diagnosticó
9. Hallazgos de optimización de la consulta (`EXPLAIN`)
10. Diccionario de datos del esquema completo
11. Costos y capa gratuita en Azure
12. Pendientes

> Este notebook asume que se abre desde la raíz del proyecto, con el
> entorno virtual `API/` activo (mismas dependencias que la API).


## 1. Arquitectura general

```
MySQL (VM en Azure, IP fija)
        │  puerto 3306
        ▼
main/database.py   → pool de conexiones (pymysql), reconexión automática
        │
main/queries.py    → arma el SQL dinámicamente (reporte + catálogos), valida filtros
        │
main/main.py       → FastAPI: expone los endpoints REST
        │
main/auth.py       → JWT (Bearer Token) — módulo listo, actualmente sin activar
        ▼
Consumidores (jobs, dashboards, Excel, Postman, etc.)
```

**Componentes clave:**

| Archivo | Responsabilidad |
|---|---|
| `main/main.py` | Define los endpoints FastAPI, valida query params, arma respuestas (JSON/CSV/Excel) |
| `main/database.py` | Pool de conexiones MySQL con `pymysql`, ping+reconexión antes de cada uso |
| `main/queries.py` | Construye el SQL del reporte y de los catálogos, con whitelist de columnas filtrables |
| `main/auth.py` | JWT — `POST /api/v1/auth/login` sigue funcionando, pero ningún endpoint lo exige hoy |
| `sql_scripts/extraccion.sql` | La consulta de referencia (ventas + devoluciones) — fuente única de verdad de la lógica de negocio |


## 2. Setup — conexión reutilizable

Esta celda se reutiliza en el resto del notebook para conectarse directo a MySQL
(igual que hace `main/database.py`, pero sin el pool — conexión simple para explorar).


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import pymysql
import requests
from dotenv import load_dotenv

RAIZ = Path.cwd()
if not (RAIZ / ".env").exists() and (RAIZ.parent / ".env").exists():
    RAIZ = RAIZ.parent  # por si el notebook se abre desde una subcarpeta

load_dotenv(RAIZ / ".env")
sys.path.insert(0, str(RAIZ / "main"))


def get_connection():
    """Conexión directa a MySQL, igual que main/database.py pero sin pool."""
    return pymysql.connect(
        host=os.getenv("DB_HOST"),
        port=int(os.getenv("DB_PORT", "3306")),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        database=os.getenv("DB_NAME"),
        cursorclass=pymysql.cursors.DictCursor,
        connect_timeout=30,
    )


print("DB_HOST configurado:", bool(os.getenv("DB_HOST")))


## 3. La consulta SQL base (`sql_scripts/extraccion.sql`)

Es un `UNION ALL` de dos bloques con la **misma estructura de columnas**:

```
BLOQUE 1: Ventas (valores positivos)
          UNION ALL
BLOQUE 2: Devoluciones (valores negativos)
```

`UNION ALL` (no `UNION`) porque no hace falta deduplicar — cada fila es una línea de
detalle distinta, y deduplicar sería más lento sin ningún beneficio.

### Bloque Ventas

```sql
FROM le.ventas v
INNER JOIN le.ventasdetalle vd    -- piezas, precio, costo por línea
INNER JOIN le.articulo a          -- descripción del producto
LEFT  JOIN le.factura f           -- factura (puede no existir)
LEFT  JOIN le.cliente c           -- cliente (llega a través de la factura)
LEFT  JOIN le.almacen am          -- sucursal
WHERE v.VentasEstado = 'Activa'
```

- `ventasdetalle` y `articulo` son **INNER**: una venta siempre tiene detalle, y el
  detalle siempre referencia un artículo válido.
- `factura`, `cliente`, `almacen` son **LEFT**: una venta puede no tener factura aún.

**El detalle técnico más importante de todo el query:**

```sql
LEFT JOIN le.factura f
        ON f.FacturaId = v.VentasFacturaId
       AND f.FacturaEstado = '0'          -- ← dentro del ON, no en el WHERE
```

Si ese filtro estuviera en el `WHERE`, cualquier venta sin factura (o con factura en
otro estado) sería excluida por completo — convirtiendo el LEFT JOIN en un INNER JOIN
de facto. Al ponerlo en el `ON`, la venta se mantiene en el reporte aunque su factura
no pase el filtro (columna `EstatusDocumento` = 'Sin factura').

### Bloque Devoluciones (el espejo, en negativo)

Misma estructura, pero:

1. Todos los valores numéricos se multiplican por `-1` — así, sumar `Total` en el
   reporte combinado da la **venta neta** real sin lógica adicional.
2. Doble filtro: `DevolucionVentaEstado = 0 AND DevolucionVentaCancelada = 0`.

### Columnas calculadas

- `Totaldecosto` = costo unitario × piezas
- `MARGENUTILIDAD` = `(venta - costo) / venta × 100`, con `NULLIF(..., 0)` para
  evitar división por cero
- `TipoMovimiento` ('Venta'/'Devolucion') — permite filtrar con `?tipo_movimiento=` en la API
- `EstatusDocumento` — indica si el movimiento tiene comprobante fiscal asociado


In [ ]:
query_ventas_ejemplo = """
SELECT
  am.AlmacenId AS IDAlmacen,
  v.VentasFecha AS fecha,
  vd.VentasDetalleCantidad AS PIEZAS,
  vd.VentasDetallePrecio AS Precio,
  vd.VentasDetalleSubTotal AS Total,
  'Venta' AS TipoMovimiento
FROM le.ventas v
INNER JOIN le.ventasdetalle vd ON v.VentasId = vd.VentasId
INNER JOIN le.articulo a ON vd.ArticuloId = a.ArticuloId
LEFT JOIN le.factura f ON f.FacturaId = v.VentasFacturaId AND f.FacturaEstado = '0'
LEFT JOIN le.cliente c ON c.ClienteId = f.FacturaClienteId
LEFT JOIN le.almacen am ON am.AlmacenId = v.VentasAlmacenId
WHERE v.VentasEstado = 'Activa'
ORDER BY v.VentasFecha DESC
LIMIT 10
"""

with get_connection() as conn:
    df_ejemplo = pd.read_sql(query_ventas_ejemplo, conn)

df_ejemplo


## 4. Cómo `main/queries.py` arma las consultas dinámicamente

`build_reporte_query(filtros)` arma **dos** subconsultas (ventas y devoluciones), cada
una con sus propios filtros de fecha/almacén/cliente aplicados **antes** del
`UNION ALL` — no como una capa externa que filtraría después de traer todo. Esto es
clave para el rendimiento (ver sección 9).

`build_catalogo_query(tabla, filtros)` arma consultas genéricas para los 8 catálogos,
validando cada filtro dinámico contra una whitelist de columnas (`CATALOG_FIELDS`) —
así se evita inyección SQL vía nombre de columna.


In [ ]:
import queries

filtros_ejemplo = {
    "fecha_inicio": "2026-08-01",
    "fecha_fin": "2026-08-31",
    "almacen_id": None,
    "cliente_id": None,
    "tipo_movimiento": None,
    "limit": 10,
    "offset": 0,
    "download_all": False,
}

sql, parametros = queries.build_reporte_query(filtros_ejemplo)
print(sql)
print("Parámetros:", parametros)


**Catálogos disponibles** (`queries.TABLAS_CATALOGO`): `almacenes`, `clientes`,
`articulos`, `facturas`, `ventas`, `ventas-detalle`, `devoluciones`,
`devoluciones-detalle`. Cada uno expone solo un subconjunto de columnas (excepto
`almacenes`, que expone la tabla completa) — ver `queries.CATALOG_FIELDS`.


## 5. El pool de conexiones (`main/database.py`)

Por qué existe un pool (y no una conexión nueva por request):

- La BD es remota (VM en Azure) — abrir una conexión TCP nueva en cada request
  tiene overhead de latencia de red + handshake.
- El pool mantiene `DB_POOL_SIZE` conexiones (default 5) ya abiertas y listas.

**El detalle importante:** antes de entregar una conexión del pool, se hace
`conn.ping(reconnect=True)` — porque la BD es remota y algún firewall/NAT intermedio
puede cerrar conexiones inactivas sin avisar. Si la conexión murió **durante** una
consulta (a media transacción), no se intenta "resucitar": se descarta, se repone el
pool, y el error se deja propagar (el cliente recibe 502 y puede reintentar el GET).


## 6. Endpoints de la API (`main/main.py`)

| Método | Ruta | Descripción |
|---|---|---|
| GET | `/` | Healthcheck simple |
| GET | `/healthz` | Health check + verifica conexión a BD |
| POST | `/api/v1/auth/login` | Obtiene JWT (no exigido hoy por ningún endpoint) |
| GET | `/api/v1/reporte-ventas-netas` | Reporte combinado (ventas + devoluciones) |
| GET | `/api/v1/reporte-ventas-netas/total` | Conteo de registros del reporte |
| GET | `/api/v1/catalogs` | Lista los 8 catálogos disponibles |
| GET | `/api/v1/catalogs/{tabla}` | Datos de un catálogo (columnas limitadas) |
| GET | `/api/v1/catalogs/{tabla}/total` | Conteo de registros de un catálogo |

**Filtros del reporte:** `fecha_inicio`, `fecha_fin`, `almacen_id`, `cliente_id`,
`tipo_movimiento` (`Venta`/`Devolucion`), `limit` (máx. 5000), `offset`, `format`
(`json`/`csv`/`excel`), `download_all` (ignora limit/offset).

Prueba en vivo (requiere la API corriendo localmente con
`uvicorn main:app --reload` desde `main/`):


In [ ]:
BASE_URL = "http://127.0.0.1:8000"

try:
    r = requests.get(f"{BASE_URL}/healthz", timeout=5)
    print(r.status_code, r.json())
except requests.exceptions.ConnectionError:
    print("La API no está corriendo localmente. Inícia con:")
    print("  cd main && ../API/Scripts/uvicorn.exe main:app --reload")


In [ ]:
# Ejemplo: reporte con filtros, formato JSON
r = requests.get(f"{BASE_URL}/api/v1/reporte-ventas-netas", params={
    "fecha_inicio": "2026-08-01",
    "fecha_fin": "2026-08-31",
    "limit": 10,
})
pd.DataFrame(r.json()) if r.status_code == 200 else r.json()


## 7. Autenticación JWT (`main/auth.py`) — desactivada

Estado actual: **ningún endpoint la exige**. El módulo sigue funcionando:

```python
POST /api/v1/auth/login?username=admin&password=admin123
→ {"access_token": "...", "token_type": "bearer"}
```

**Para reactivarla:** agregar `Depends(auth.verify_token)` a los endpoints en
`main.py`. Al hacerlo, cualquier consumidor de la API (incluyendo los 8
clientes/jobs actuales) deberá empezar a mandar el header
`Authorization: Bearer <token>` — hay que coordinar ese cambio con quien consume
la API, no es solo un cambio de servidor.

Usuarios por defecto: `admin`/`admin123`, `api_user`/`api_pass_123` (cambiar antes
de usar en producción real).


## 8. Pipeline de despliegue: Docker → GitHub Actions → GHCR → Azure Container Apps

### 8.1 Dockerfile (multi-stage)

```dockerfile
FROM python:3.11-slim AS builder
RUN apt-get update && apt-get install -y --no-install-recommends gcc && rm -rf /var/lib/apt/lists/*
COPY requirements.txt .
RUN pip install --no-cache-dir --prefix=/install -r requirements.txt

FROM python:3.11-slim
COPY --from=builder /install /usr/local
COPY main/ /app/main/
COPY sql_scripts/ /app/sql_scripts/
WORKDIR /app/main
RUN useradd -m -u 1000 appuser && chown -R appuser:appuser /app
USER appuser
EXPOSE 8000
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

**Dos bugs reales que se encontraron y corrigieron durante el desarrollo** (vale la
pena recordarlos si se vuelve a tocar el Dockerfile):

1. ❌ `pip install --wheel --wheel-dir ...` — `--wheel-dir` es una opción de
   `pip wheel`, no de `pip install`. Causaba `exit code: 2`.
2. ❌ El patrón inicial de `pip wheel --no-deps` + `pip install --no-index` omitía
   las **dependencias transitivas** de cada paquete (ej. lo que FastAPI necesita
   internamente) → `exit code: 1` en la segunda etapa. Se resolvió cambiando a
   `pip install --prefix=/install` (resuelve todo con acceso normal a PyPI) +
   copiar ese prefix a `/usr/local` en la imagen final.

### 8.2 GitHub Actions (`.github/workflows/build-and-push.yml`)

Construye la imagen y la publica en **GitHub Container Registry (GHCR)** — gratis,
en vez de Azure Container Registry (~$5 USD/mes fijos sin nivel gratuito).

Bug encontrado: `cache-to: type=gha` requiere el driver `docker-container` de
Buildx, no el driver `docker` por defecto → hubo que agregar
`docker/setup-buildx-action@v3` antes del paso de build.

### 8.3 Azure Container Apps

```bash
az containerapp create \
  --name api-extraccion --resource-group rg-api-extraccion \
  --environment env-api-extraccion \
  --image "ghcr.io/alejos94/mikels-api-lidex-extraccion-datos:latest" \
  --target-port 8000 --ingress external \
  --min-replicas 0 --max-replicas 3 --cpu 0.5 --memory 1.0Gi \
  --secrets db-host=... db-port=3306 db-user=... db-password=... db-name=... \
  --env-vars DB_HOST=secretref:db-host DB_PORT=secretref:db-port ...
```

**`--min-replicas 0` es la clave del costo $0** — escala a cero sin tráfico. El costo
de tenerlo en 0 es un cold start de 1-2s en la primera petición tras inactividad.


## 9. El problema de red (NSG) — cómo se diagnosticó

**Síntoma:** la API desplegada respondía bien en `/` y `/api/v1/catalogs`, pero
`/healthz` y los endpoints con BD daban `502` con timeout al conectar a MySQL.

**Diagnóstico paso a paso:**

1. Confirmar que las credenciales eran correctas (se corrigieron placeholders
   olvidados) → seguía fallando, ahora con **timeout** (no "acceso denegado") →
   apunta a bloqueo de red, no de credenciales.
2. Verificar el Firewall de Windows en la VM → **apagado** para el perfil "Público"
   (que es el activo) → descarta Windows Firewall como bloqueo.
3. Verificar el propietario de la IP del servidor MySQL vía WHOIS/RDAP
   (`ipinfo.io`, ARIN) → confirmado: rango de Microsoft/Azure (AS8075).
4. Conclusión: el bloqueo es un **NSG de Azure**, en una suscripción distinta a la
   usada para el despliegue (`az network nsg list` no devolvía nada).

**Por qué pasa esto en cualquier nube (no es exclusivo de Azure):** Container Apps
(y sus equivalentes App Runner en AWS, Cloud Run en GCP) usan un **pool compartido
de IPs de salida dinámicas** — no una IP fija — salvo que se pague por integración a
VPC/VNET + NAT Gateway (~$32-45 USD/mes extra). Por eso el firewall de la VM, que
solo permite IPs conocidas, rechaza las conexiones.

**Fix pendiente/aplicado:** agregar una regla de entrada en el NSG:

```bash
az network nsg rule create \
  --resource-group <grupo-de-la-vm> --nsg-name <nombre-nsg> \
  --name Allow-AzureCloud-MySQL --priority 200 \
  --source-address-prefixes AzureCloud.EastUS \
  --destination-port-ranges 3306 --access Allow --protocol Tcp --direction Inbound
```


## 10. Hallazgos de optimización (vía `EXPLAIN`)

| Prueba | Resultado |
|---|---|
| `ventas` con filtro de fecha | ✅ Usa índice `UVENTASFECHA` (`type: range`) — bien optimizado |
| Todos los LEFT JOIN (factura, cliente, almacén, artículo) | ✅ `eq_ref` vía llave primaria — el tipo de acceso más rápido posible |
| `devolucionventa` con filtro de fecha | ❌ `type: ALL` — **sin índice en `DevolucionVentaFecha`** |
| Consulta completa (`UNION ALL` + `ORDER BY fecha DESC LIMIT`) | ❌ `Using temporary; Using filesort` — el `LIMIT` no evita ordenar todo el resultado combinado |
| Sin filtro de fecha (caso por defecto de la API) | ❌ El optimizador cambia el plan por completo, arranca desde `articulo` en vez de `ventas`, examinando ~51,000 filas — el `LIMIT` no ayuda porque no puede aprovechar el índice de fecha |

**Recomendaciones (pendientes de aplicar, decisión del usuario):**

```sql
-- Bajo riesgo, aditivo, no cambia comportamiento:
CREATE INDEX IX_DevolucionVentaFecha ON le.devolucionventa (DevolucionVentaFecha);

-- Podría ayudar al optimizador a elegir el plan correcto sin fecha:
CREATE INDEX IX_VentasEstado_Fecha ON le.ventas (VentasEstado, VentasFecha);
```

**Bug de consistencia encontrado** (no solo performance): `build_count_query` en
`queries.py` no replica el filtro `FacturaEstado = '0'` en su LEFT JOIN a `factura`
que sí tiene `build_reporte_query`. Esto puede causar que
`/api/v1/reporte-ventas-netas/total` **no coincida** con la cantidad real de filas
que devuelve `/api/v1/reporte-ventas-netas` cuando se filtra por `cliente_id`.
Pendiente de corregir si se decide.


## 11. Diccionario de datos del esquema completo

En `data_dictionary/` (uso personal, fuera de git):

| Archivo | Qué es |
|---|---|
| `generar_diccionario.py` | Introspecciona `INFORMATION_SCHEMA` de las 207 tablas del esquema `LE` |
| `enriquecer.py` | Agrega contexto de negocio a las 8 tablas ya expuestas por la API |
| `compactar.py` | Genera la versión optimizada + heurísticas genéricas por nombre de columna |
| `schema.yml` | Formato **dbt** — reutilizable directo para el futuro pipeline hacia Snowflake |
| `visor.html` | Página interactiva (publicada como Artifact) para buscar/navegar las 207 tablas |

**Estado de documentación:** 83 columnas con descripción de negocio real, 854 con
heurística genérica por nombre, 2,014 sin documentar — el trabajo pendiente real
antes de construir el Data Lakehouse.


## 12. Costos y capa gratuita en Azure

Con el volumen actual (8 conexiones × 9 endpoints/día ≈ 2,160 requests/mes):

| Componente | Costo |
|---|---|
| Cómputo Container Apps (`min-replicas=0`) | $0 (muy por debajo de 180K vCPU-seg / 360K GiB-seg gratis/mes) |
| Requests | $0 (muy por debajo de 2M gratis/mes) |
| Registro de imágenes (GHCR en vez de ACR) | $0 |
| GitHub Actions (build ~3-5 min por cambio de código) | $0 (repo privado: 2,000 min/mes gratis) |
| **Total estimado** | **$0/mes** |

**Lo que sí cuesta dinero (evitar mientras el uso sea bajo):**
- `--min-instances=1` (instancia siempre activa): ~$65 USD/mes extra — solo vale la
  pena si el volumen de requests crece mucho o se necesita respuesta garantizada sin
  cold start.
- VNET + NAT Gateway para IP de salida fija: ~$32-45 USD/mes extra — alternativa más
  segura a abrir el NSG a todo el rango `AzureCloud`, pero no necesaria a este nivel
  de uso.


## 13. Pendientes

- [ ] Confirmar que TI aplicó la regla del NSG (`AzureCloud.EastUS`, puerto 3306)
- [ ] Decidir si se crea el índice `IX_DevolucionVentaFecha` (bajo riesgo)
- [ ] Decidir si se corrige el bug de `build_count_query` (discrepancia con `cliente_id`)
- [ ] Decidir cuándo reactivar autenticación JWT (coordinar con consumidores de la API)
- [ ] Seguir llenando `descripcion_negocio` en el diccionario de datos (2,014 columnas pendientes)
- [ ] Evaluar migrar el repo de GitHub de la cuenta personal (`Alejos94`) a una organización de Mikels
- [ ] Cambiar la contraseña de MySQL que se compartió en texto plano durante las pruebas de esta sesión
- [ ] Definir el diseño del Data Lakehouse (Snowflake) usando `schema.yml` como base
